# Build the Celeb-DF v2 test set

Crops the faces of the 518 official Celeb-DF v2 test videos with the same RetinaFace pipeline as `build_dataset.ipynb`, and uploads them as a **private** Hugging Face dataset. The lab never trains on Celeb-DF: these crops are the test side of the `cross_celebdf` protocol, which measures how a detector trained on DeepFakeFace and AxonData handles face swaps from another dataset and pipeline.

**Access:** Celeb-DF is shared per researcher by its authors, for non-commercial research only and without redistribution. Ask for it through the form on the [Celeb-DF repository](https://github.com/yuezunli/celeb-deepfakeforensics), then copy `Celeb-DF-v2.zip` into your own Google Drive (a copy made in Drive avoids the shared file's download quota). Nothing here downloads the archive from anywhere else.

**Runtime:** Colab Pro with a GPU. **Needs** the Colab secret `HF_TOKEN` with write access, for the upload cell.

### Configuration

In [ ]:
REPO_SOURCE = "git+https://github.com/paulohenriquevn/face-decision-lab@workspace"
# ALIGN = True crops the way GenD does: 32 frames per video, each face warped onto 5 reference
# landmarks with a 1.3x margin (faces.aligned_crop). False keeps the lab's square 1.5x crop.
ALIGN = True
HF_DATASET = "paulohenriquevn/face-decision-celebdf-test" + ("-aligned" if ALIGN else "")  # private
ARCHIVE = "/content/drive/MyDrive/face-decision-lab/Celeb-DF-v2.zip"  # your own copy
BUILD_DIR = "/content/build_celebdf" + ("_aligned" if ALIGN else "")
VIDEO_DIR = "/content/celebdf_videos"
FRAMES_PER_VIDEO = 32 if ALIGN else 16

### Installation

Same as `build_dataset.ipynb`: `onnxruntime-gpu` and the ONNX backend, so RetinaFace runs on the GPU.

In [ ]:
%%capture
!pip install -q "face-decision-lab[build] @ {REPO_SOURCE}"
!pip uninstall -y -q onnxruntime
!pip install -q onnxruntime-gpu

In [ ]:
import os
os.environ["DEEPFACE_BACKEND_ENGINE"] = "onnx"

import onnxruntime as ort
providers = ort.get_available_providers()
print("ONNX Runtime providers:", providers)
assert "CUDAExecutionProvider" in providers, "onnxruntime-gpu cannot see the GPU; RetinaFace would run on the CPU"

### Extract the official test videos

Reads only the 518 test videos out of the archive in Drive. `official_test_videos` checks every label of `List_of_testing_videos.txt` against the folder the video sits in.

In [ ]:
import zipfile
from pathlib import Path
from google.colab import drive
from face_decision_lab.sources import celebdf

drive.mount("/content/drive")
video_dir = Path(VIDEO_DIR)
with zipfile.ZipFile(ARCHIVE) as archive:
    test_videos = sorted(celebdf.official_test_videos(archive.read(celebdf.TEST_LIST).decode()))
    for name in test_videos:
        if not (video_dir / name).exists():
            archive.extract(name, video_dir)
items = [celebdf.parse_member(name) for name in test_videos]
print(len(items), "test videos")

### Build

Resumable, like the main build: a rerun skips every video already in `processed.jsonl`.

In [ ]:
from tqdm.auto import tqdm
from face_decision_lab.build import DatasetBuilder
from face_decision_lab.faces import RetinaFaceDetector, aligned_crop, square_crop

cropper = (lambda image, box: aligned_crop(image, box.landmarks)) if ALIGN else square_crop
builder = DatasetBuilder(BUILD_DIR, RetinaFaceDetector(), frames_per_video = FRAMES_PER_VIDEO, cropper = cropper)
for item in tqdm(items, desc = "celebdf"):
    builder.add_video(item, video_dir / item.path)
builder.flush()
print(len(builder.samples), "crops from", len({s.video_id for s in builder.samples}), "videos")

### Check detection outcomes

A detector that drops fakes more often than real videos biases the test. Compare the saved share per generator before uploading.

In [ ]:
for generator in sorted(builder.report.counts):
    counts = builder.report.counts[generator]
    print(f"{generator:24s} saved {builder.report.saved_share(generator):6.1%}  {counts}")

### Upload as a private dataset

In [ ]:
from google.colab import userdata
from huggingface_hub import login
from face_decision_lab.hub import push_build

login(token = userdata.get("HF_TOKEN"))
print(push_build(BUILD_DIR, HF_DATASET))